# 01 — Environment Exploration

This notebook walks through setting up the ViZDoom environment, inspecting
observations, testing wrappers (frame stacking, resizing), and running a
random-agent baseline.

## 1. Install & verify dependencies

In [ ]:
# Uncomment the line below when running on Colab
# !pip install vizdoom gymnasium torch opencv-python matplotlib

import vizdoom
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

print(f"ViZDoom version: {vizdoom.__version__}")

In [ ]:
# Google Drive integration for persistent storage on Colab
# Uncomment the lines below when running on Google Colab
# import os
# from google.colab import drive
# drive.mount('/content/drive')
# DRIVE_ROOT = "/content/drive/MyDrive/rl-doom"
# os.makedirs(DRIVE_ROOT, exist_ok=True)
# for subdir in ["checkpoints", "logs", "figures", "media"]:
#     os.makedirs(f"{DRIVE_ROOT}/{subdir}", exist_ok=True)
# # Symlink so relative paths (../checkpoints, ../logs, etc.) resolve to Drive
# for subdir in ["checkpoints", "logs", "figures", "media"]:
#     local = os.path.abspath(f"../{subdir}")
#     if not os.path.exists(local):
#         os.symlink(f"{DRIVE_ROOT}/{subdir}", local)
# print(f"Google Drive mounted. Artifacts will persist at: {DRIVE_ROOT}")

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

from rl_doom.env import DoomEnv, FrameStack, ResizeObservation, SkipFrame

## 2. Create a basic environment

We start with the **Basic** scenario — a single room where the agent must
shoot a monster as quickly as possible.

In [ ]:
env = DoomEnv(scenario="basic")
obs, info = env.reset(seed=42)

print(f"Observation shape : {obs.shape}")
print(f"Observation dtype : {obs.dtype}")
print(f"Action space      : {env.action_space}")
print(f"Available actions : {env.available_actions}")

## 3. Visualize raw observations

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))

obs, info = env.reset(seed=42)
axes[0].imshow(obs)
axes[0].set_title("After reset")

for i in range(1, 4):
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    axes[i].set_title(f"Step {i} (a={action}, r={reward:.1f})")
    axes[i].imshow(obs)

for ax in axes:
    ax.axis("off")
plt.tight_layout()
os.makedirs("../figures", exist_ok=True)
plt.savefig("../figures/01_raw_observations.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Test wrappers

Apply **ResizeObservation** (84×84 grayscale), **SkipFrame** (repeat=4), and
**FrameStack** (4 frames) — the standard Atari-style preprocessing pipeline.

In [ ]:
env_wrapped = DoomEnv(scenario="basic")
env_wrapped = ResizeObservation(env_wrapped, shape=(84, 84))
env_wrapped = SkipFrame(env_wrapped, skip=4)
env_wrapped = FrameStack(env_wrapped, num_stack=4)

obs, info = env_wrapped.reset(seed=42)
print(f"Wrapped observation shape: {obs.shape}  (expect [4, 84, 84])")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
for i in range(4):
    axes[i].imshow(obs[i], cmap="gray")
    axes[i].set_title(f"Frame {i}")
    axes[i].axis("off")
plt.suptitle("Stacked grayscale frames after reset", y=1.02)
plt.tight_layout()
plt.savefig("../figures/01_stacked_frames.png", dpi=150, bbox_inches="tight")
plt.show()

## 5. Random-agent baseline

Run a random policy for multiple episodes and record total rewards. This gives
us a lower-bound to compare against trained agents.

In [ ]:
N_EPISODES = 50
episode_rewards = []

for ep in range(N_EPISODES):
    obs, info = env_wrapped.reset(seed=ep)
    total_reward = 0.0
    done = False
    while not done:
        action = env_wrapped.action_space.sample()
        obs, reward, terminated, truncated, info = env_wrapped.step(action)
        total_reward += reward
        done = terminated or truncated
    episode_rewards.append(total_reward)

episode_rewards = np.array(episode_rewards)
print(f"Random agent — {N_EPISODES} episodes")
print(f"  Mean reward: {episode_rewards.mean():.2f} ± {episode_rewards.std():.2f}")
print(f"  Min / Max  : {episode_rewards.min():.2f} / {episode_rewards.max():.2f}")

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(episode_rewards, alpha=0.6, label="Episode reward")
plt.axhline(episode_rewards.mean(), color="red", linestyle="--", label="Mean")
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.title("Random Agent — Basic Scenario")
plt.legend()
plt.tight_layout()
plt.savefig("../figures/01_random_baseline.png", dpi=150, bbox_inches="tight")
plt.show()

# Save baseline stats for later comparison
os.makedirs("../logs", exist_ok=True)
np.savez(
    "../logs/random_baseline_basic.npz",
    rewards=episode_rewards,
    mean=episode_rewards.mean(),
    std=episode_rewards.std(),
)

## 6. Explore other scenarios

In [ ]:
scenarios = ["basic", "deadly_corridor", "defend_the_center", "deathmatch"]

for scenario in scenarios:
    try:
        e = DoomEnv(scenario=scenario)
        obs, _ = e.reset()
        print(f"{scenario:20s}  obs={obs.shape}  actions={e.action_space.n}")
        e.close()
    except Exception as ex:
        print(f"{scenario:20s}  ERROR: {ex}")

In [ ]:
env.close()
env_wrapped.close()
print("Done!")